In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

# Состав выбран жадным отбором на честном фолде: 0.6642 против 0.6361 у пары
# relaxed+combo, которая сейчас отправлена. Отбор шёл по разнообразию, а не по силе:
# spec самая независимая из шести кросс-энкодеров, e5 другая основа, hardneg другие
# негативы, bi другая архитектура.
ENCODERS = ("ce_spec", "ce_e5", "ce_hardneg", "ce_bi")

fold_dir = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
v2 = os.path.dirname(glob.glob("/kaggle/input/**/ce_balanced.npy", recursive=True)[0])
# `ce_bi.npy` лежит в двух местах: в датасете с оценками на фолде (156 032 значения) и в
# выгрузке ядра оценок на новых парах (400 000). Раньше путь брался первым попавшимся из
# glob, и обучение получало длину не от тех пар. Датасеты монтируются под
# /kaggle/input/datasets, выгрузки ядер — под /kaggle/input/notebooks, по этому и разводим.
bi_hits = [h for h in glob.glob("/kaggle/input/datasets/**/ce_bi.npy", recursive=True)]
if len(bi_hits) != 1:
    raise SystemExit(f"оценки двухбашенной на фолденеоднозначны: {bi_hits}")
bi_dir = os.path.dirname(bi_hits[0])
hn_hits = glob.glob("/kaggle/input/**/ce_hardneg/llm_ood_scores.npy", recursive=True)
extra = os.path.dirname(glob.glob("/kaggle/input/**/clean_pairs_v4.parquet", recursive=True)[0])
log(f"фолд: {fold_dir}\n  скоры: {v2}\n  контрастная: {bi_dir}\n  негативы: {hn_hits}\n  новые пары: {extra}")

vp = pd.read_parquet(fold_dir + "/llm_valid_pairs.parquet")
vi = pd.read_parquet(fold_dir + "/llm_valid_items.parquet")
KNOWN = sorted(set(vi.category.astype(str)))
COLUMNS_TAIL = list(ENCODERS) + ["structural", "category_code"]

def scores_for_fold():
    out = {}
    out["ce_spec"] = np.load(f"{v2}/ce_spec.npy").astype(np.float32)
    out["ce_e5"] = np.load(f"{v2}/ce_e5.npy").astype(np.float32)
    out["ce_bi"] = np.load(f"{bi_dir}/ce_bi.npy").astype(np.float32)
    full = np.load(hn_hits[0]).astype(np.float32)
    src = pd.read_parquet(os.path.dirname(hn_hits[0]) + "/llm_ood_pairs.parquet")
    pos = pd.Series(np.arange(len(src)), index=pd.MultiIndex.from_arrays([src.id1, src.id2]))
    take = pos.reindex(pd.MultiIndex.from_arrays([vp.id1, vp.id2])).to_numpy()
    if not np.isfinite(take).all():
        raise SystemExit("скоры негативов не покрывают фолд")
    out["ce_hardneg"] = full[take.astype(int)]
    return out

def prepare(pairs, items, scores, tag):
    y = ((pairs["target"] if "target" in pairs else pairs["label"]).to_numpy() > 0).astype(np.int8)
    cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
    names = list(feature_names(False, True))
    X = np.zeros((len(pairs), len(names)), dtype=np.float32)
    for c in sorted(set(items.category.astype(str))):
        rows = np.flatnonzero(cat == c)
        if not len(rows): continue
        sub = items[items.category.astype(str) == c].reset_index(drop=True)
        X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), KNOWN,
                               with_neighbours=False, with_measures=True)
        del sub; gc.collect()
    legacy = extract_model_features(pairs[["id1", "id2"]], items)
    pr = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
    au = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
    del legacy; gc.collect()
    codes = np.array([KNOWN.index(c) if c in KNOWN else -1 for c in cat], dtype=np.float32)
    for e in ENCODERS:
        if len(scores[e]) != len(pairs):
            raise SystemExit(f"{tag}: у {e} {len(scores[e]):,} оценок на {len(pairs):,} пар")
    M = np.column_stack([X] + [scores[e] for e in ENCODERS] + [0.8*pr + 0.2*au, codes])
    log(f"  {tag}: {M.shape}, доля+ {y.mean():.3f}")
    return M.astype(np.float32), y, cat

Mv, yv, catv = prepare(vp, vi, scores_for_fold(), "фолд")
ep = pd.read_parquet(extra + "/clean_pairs_v4.parquet")
ei = pd.read_parquet(extra + "/clean_items_v4.parquet")
Me, ye, _ = prepare(ep, ei, {e: np.load(f"{extra}/{e}.npy").astype(np.float32) for e in ENCODERS},
                    "новые пары")

masks = {c: catv == c for c in np.unique(catv)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos_, neg = rows[yv[rows] == 1], rows[yv[rows] == 0]
            keep = min(len(pos_), max(5, int(round(rate / (1 - rate) * len(neg)))))
            ch = np.concatenate([rng.choice(pos_, keep, replace=False), neg])
            per.append(average_precision_score(yv[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(yv)) % 2
oof_small = np.zeros(len(yv)); oof_big = np.zeros(len(yv))
for h in (0, 1):
    tr, te = half != h, half == h
    oof_small[te] = HistGradientBoostingClassifier(**PARAMS).fit(Mv[tr], yv[tr]).predict_proba(Mv[te])[:, 1]
    big = HistGradientBoostingClassifier(**PARAMS).fit(np.vstack([Mv[tr], Me]),
                                                       np.concatenate([yv[tr], ye]))
    oof_big[te] = big.predict_proba(Mv[te])[:, 1]
a, sa = macro(rk(oof_small)); b, sb = macro(rk(oof_big))
log(f"только фолд ({len(yv):,}):        {a:.6f} ± {sa:.6f}")
log(f"фолд + новые ({len(yv)+len(ye):,}): {b:.6f} ± {sb:.6f}   ({b-a:+.6f})")

use_extra = b > a
Xall = np.vstack([Mv, Me]) if use_extra else Mv
yall = np.concatenate([yv, ye]) if use_extra else yv
final = HistGradientBoostingClassifier(**PARAMS).fit(Xall, yall)
trees = export(final)
save("/kaggle/working/fusion_boost.npz", trees)
ok = np.allclose(predict_proba(trees, Xall[:2000]), final.predict_proba(Xall[:2000])[:, 1], atol=1e-6)
COLS = list(feature_names(False, True)) + COLUMNS_TAIL
json.dump({"columns": COLS, "categories": KNOWN, "params": PARAMS,
           "honest_macro": max(a, b), "n_train": int(len(yall)),
           "uses_structural": True, "encoders": list(ENCODERS),
           "biencoder": "ce_bi", "used_extra_pairs": bool(use_extra)},
          open("/kaggle/working/fusion_info.json", "w"), ensure_ascii=False, indent=1)
log(f"выгружено: {len(COLS)} столбцов, обучено на {len(yall):,}, совпадение {ok}")
